# Rebuild the data on Google Colab

The repo already ships the finished data in `data/`, so you only need this
notebook to rebuild it (new dataset version, different filter, better
translation model).

Before running: **Runtime > Change runtime type > T4 GPU**. Switching later
wipes the files. With the GPU the whole notebook takes about 10 minutes.

In [ ]:
!git clone -q https://github.com/MahmudulHuda/Alzheimers-Dementia-Care-Assistant.git /content/app
%cd /content/app
!pip install -q -r requirements.txt -r requirements-translate.txt
!nvidia-smi -L || echo "No GPU - translation will be very slow"

## 1. Download MedQuAD and keep the Alzheimer's / dementia part

In [ ]:
!python scripts/fetch_data.py

## 2. Translate to Bangla and Chinese

Google Translate blocks Colab, so this uses the offline NLLB-200 model.
The first run downloads it (about 2.5 GB). Try `--limit 3` first if you like.

In [ ]:
!python scripts/translate_nllb.py

## 3. Try it

In [ ]:
from careqa.assistant import Assistant
bot = Assistant()
for q in ["what are the early signs of alzheimer's",
          "আলঝেইমার রোগের লক্ষণ কী?",
          "阿尔茨海默病有哪些症状？"]:
    r = bot.ask(q)
    print(q, "->", r.get("matched"), "\n", r["answer"][:200], "\n")

## 4. Save the results

Colab deletes everything when the session ends. Download the data and copy
the files into `data/` in your local copy of the repo.

In [ ]:
!zip -q data.zip data/kb_en.json data/kb_bn.json data/kb_zh.json
from google.colab import files
files.download("data.zip")